# 01 — Sổ Tay EDA: Phễu GA4 8 Bước & Vùng Giá Ngọt Trúng Thầu (`goodNstay`)

**Mục đích (Executable Lab Notebook cho đội Data/PM & CTO):**
1. Đọc dữ liệu chuẩn hóa từ **Google BigQuery (`aevo-512307.gns_analytics_mart`)** có khóa cứng chi phí `BQ_MAX_BYTES_BILLED` (tự động fallback về `seed/bid_records.json` nếu chạy offline).
2. Đo lường hiệu suất **Phễu 8 bước** từ `booking_form` → `offer_listing` → `explore_stays` → `payment_form`.
3. Kiểm chứng giả thuyết **Vùng giá ngọt tiết kiệm (`10%–15%`)** so với sàn đặt phòng ngoài (OTA) và tốc độ quyết định của du khách.
4. Ghi chú nhận định định kỳ phục vụ cuộc họp tối ưu thuật toán cùng CTO.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
NOTEBOOK_ROOT = next(
    (p for p in (cwd, cwd.parent, cwd / "analytics-notebooks") if (p / "src").is_dir()),
    cwd,
)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.bq_client import GoodNstayAnalyticsLoader
from src.config import load_settings

settings = load_settings()
loader = GoodNstayAnalyticsLoader(settings)
pd.Series(settings.summary_dict(), name="active_config")

## 1. Nạp KPI Theo Ngày Từ BigQuery Gold Mart (Có Giới Hạn Trần Chi Phí)

In [ ]:
START_DATE = "2026-10-08"
END_DATE = "2026-11-06"

df_daily, daily_meta = loader.load_daily_kpis(start_date=START_DATE, end_date=END_DATE)
print("Telemetry:", daily_meta)

branch_summary = (
    df_daily.groupby("branch_id", as_index=False)
    .agg(
        total_bids=("total_bids", "sum"),
        accepted_paid_bids=("accepted_paid_bids", "sum"),
        rejected_bids=("rejected_bids", "sum"),
        expired_bids=("expired_bids", "sum"),
        hotel_net_revenue_vnd=("hotel_net_revenue_vnd", "sum"),
        avg_saving_pct=("avg_saving_pct", "mean"),
        avg_match_rate_pct=("avg_match_rate_pct", "mean"),
    )
)
branch_summary["win_rate_pct"] = (
    branch_summary["accepted_paid_bids"] / branch_summary["total_bids"].clip(lower=1) * 100.0
).round(2)
branch_summary.sort_values("hotel_net_revenue_vnd", ascending=False)

## 2. Trực Quan Hóa Phễu Hành Vi GA4 8 Bước

In [ ]:
funnel_cols = [
    ("B1. Mở form tìm phòng", "step1_form_starts"),
    ("B2. Gửi yêu cầu Tìm bid", "step2_requests_submitted"),
    ("B3. Thẻ báo giá hiển thị", "step3_offers_impressed"),
    ("B4. Xem chi tiết duy nhất", "step4_unique_viewed_offers"),
    ("B5. Tương tác ảnh/video", "step5_media_engaged_offers"),
    ("B6. Bấm Chọn báo giá", "step6_unique_selected_offers"),
    ("B7. Mở trang thanh toán", "step7_checkout_opened_offers"),
    ("B8. Thanh toán hoàn tất", "step8_paid_completed_offers"),
]
funnel_df = pd.DataFrame(
    [{"step": label, "count": int(df_daily[col].sum())} for label, col in funnel_cols if col in df_daily.columns]
)
top_count = max(int(funnel_df["count"].iloc[0]), 1)
funnel_df["retention_pct"] = (funnel_df["count"] / top_count * 100.0).round(1)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(funnel_df["step"][::-1], funnel_df["count"][::-1], color="#1D4ED8")
ax.set_title(f"goodNstay 8-Step Conversion Funnel ({START_DATE} -> {END_DATE})")
ax.set_xlabel("Số lượt / báo giá")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
funnel_df

## 3. Kiểm Chứng Vùng Giá Ngọt (`saving_pct` vs. Tỷ Lệ Trúng Thầu)

In [ ]:
df_bids = loader.load_bid_telemetry_sample()
df_bids["is_won"] = (df_bids["status"] == "accepted").astype(int)
df_bids["saving_band"] = pd.cut(
    df_bids["saving_pct"],
    bins=[-1, 5, 10, 15, 100],
    labels=["<5%", "5%–10%", "10%–15% (Sweet Spot)", ">15%"],
)

band_stats = (
    df_bids.groupby("saving_band", observed=False)
    .agg(
        bids=("request_code", "count"),
        won=("is_won", "sum"),
        avg_decision_sec=("decision_duration_sec", "mean"),
    )
    .reset_index()
)
band_stats["win_rate_pct"] = (band_stats["won"] / band_stats["bids"].clip(lower=1) * 100.0).round(2)
band_stats["avg_decision_sec"] = band_stats["avg_decision_sec"].round(1)
band_stats

## 4. Nhật Ký Ghi Chú Phân Tích (CTO / Product Notes)
- **Ghi chú nguồn dữ liệu:** Kiểm tra trường `daily_meta.source` (`bigquery_live` trên `aevo-512307` hoặc `local_seed_fallback`).
- **Ghi chú tiếp theo cho Giai đoạn ML (Năm 2–3):** Dữ liệu phễu 8 bước và `saving_band` là đầu vào gốc (*first-party ground truth*) để sau này tự huấn luyện mô hình dự đoán xác suất trúng thầu (LightGBM / BigQuery ML) nhằm giảm phụ thuộc OpenAI API.